In [ ]:
# Environment configuration — replace the placeholder in the Databricks widget.
dbutils.widgets.text("storage_account", "<YOUR_STORAGE_ACCOUNT>")
storage_account = dbutils.widgets.get("storage_account")
if not storage_account or storage_account.startswith("<"):
    raise ValueError("Set the storage_account widget before running.")


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
df = spark.read.format("parquet")\
            .load(f"abfss://bronze@{storage_account}.dfs.core.windows.net/products")


In [ ]:
df.display()

In [ ]:
df = df.drop("_rescued_data")

In [ ]:
display(df)

###  Register a temporary Table view of Products to use as Function

In [ ]:
df.createOrReplaceTempView("products")

In [ ]:
%sql 
CREATE OR REPLACE FUNCTION databricks_etl_catalog.bronze.discount_price_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.90

In [ ]:
%sql
select product_id, price, databricks_etl_catalog.bronze.discount_price_func(price) as discounted_price 
from products

### **Using expr function in sql we can add the newly created function directly to the products data table**

In [ ]:
df = df.withColumn("discounted_price",expr("databricks_etl_catalog.bronze.discount_price_func(price)"))
df.display()


### **Creating other functions using Python**
### ****

In [ ]:
%sql DROP FUNCTION IF EXISTS databricks_etl_catalog.bronze.upper_func

In [ ]:
%sql 
CREATE OR REPLACE FUNCTION databricks_etl_catalog.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS 
$$ 
  return p_brand.upper() 
$$

In [ ]:
%sql
select product_id, brand, databricks_etl_catalog.bronze.upper_func(brand) as brand_upper
FROM products

In [ ]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("path", f"abfss://silver@{storage_account}.dfs.core.windows.net/products")\
    .save()

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS databricks_etl_catalog.silver.products_silver
USING DELTA
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/products'

In [ ]:
%sql
select * from databricks_etl_catalog.silver.products_silver